# Sanity 6: Repeated Token with Positional Distinction
Tests repeated-token sequence behavior with learned positional embeddings intact.

In [ ]:
import torch
from homeomorphism import models, jacobian

device = 'cuda' if torch.cuda.is_available() else 'cpu'
m = models.load_model('gpt2', weights='trained', device=device, seed=0)
sub = models.sublayer(m, 0, 'attn')

tok = int(m.tokenizer.encode('the')[0])
input_ids = torch.tensor([[tok, tok, tok, tok]], device=device)
position_ids = torch.arange(4, device=device).unsqueeze(0)
attn_mask = torch.ones_like(input_ids, device=device)

captured = {}
hndl = m.model.get_submodule(sub.hook_path).register_forward_hook(lambda _m, i, _o: captured.setdefault('h', i[0].detach()))
with torch.no_grad():
    m.model(input_ids=input_ids, attention_mask=attn_mask, position_ids=position_ids)
hndl.remove()

h = captured['h'][0].to(torch.float32).cpu()
bj, diag = jacobian.build_jacobian(sub.phi, h, scope='diagonal', evaluate='per_diagonal_slogdet')
conds = []
for i in range(h.shape[0]):
    sv = bj.svdvals(i, i)
    cond = float(sv[0] / sv[-1]) if float(sv[-1]) > 0 else float('inf')
    conds.append(cond)

print('device:', device)
print('per-token log|det|:', [float(x[1]) for x in diag])
print('per-token condition numbers:', conds)